# Around the Globe

A geography quiz with four answers and one correct option. Harder questions use wrong answers that are more similar to the correct country. The score gives more weight to a harder question.

This notebook only calls the `globe_quiz` package. The loading, similarity, questions, charts, and score live in that package.

In [ ]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import Image, display

root = Path.cwd()
if not (root / "src" / "globe_quiz").exists():
    root = root.parent
sys.path.insert(0, str(root / "src"))

from globe_quiz.cleaning import build_country_table
from globe_quiz.io_worldbank import load_indicators
from globe_quiz.plotting import (
    figures_dir,
    plot_countries_by_region,
    plot_difficulty_bands,
    plot_missing_indicators,
)
from globe_quiz.quiz import generate_question
from globe_quiz.scoring import session_score
from globe_quiz.similarity import country_similarity

## Country table

The country file is joined to World Bank population, GDP, GDP per capita, and life expectancy. A country stays in the quiz only when every indicator has a value. The year of each indicator is the latest year that is not empty, so one country can have a population from 2025 and a GDP from an earlier year.

Kosovo is `UNK` in the country file and `XKX` at the World Bank. The table keeps `UNK`, because that is the code used in the border lists.

In [ ]:
built = build_country_table()
countries = built.countries
display(built.report)
france = countries.set_index("cca3").loc["FRA"]
print(
    "France population year:", int(france["population_year"]),
    "| life expectancy year:", int(france["life_expectancy_year"]),
    "| GDP per capita year:", int(france["gdp_per_capita_year"]),
)
countries[["name", "region", "capital", "population", "population_year", "life_expectancy", "life_expectancy_year"]].head()

## Similarity

Every pair of countries gets a score from 0 to 1. It is a weighted sum of five parts: distance and subregion, GDP per capita, population and life expectancy, shared languages, and a shared currency. A country compared with itself scores 1.

In [ ]:
scores = country_similarity(countries).scores
names = countries.set_index("cca3")["name"]
for left, right in (("FRA", "BEL"), ("FRA", "DEU"), ("FRA", "JPN"), ("FRA", "AUS")):
    print(f"{names.loc[left]} - {names.loc[right]}: {scores.loc[left, right]:.3f}")

## Questions

A question has four options and one correct answer. The three wrong answers come from other countries. Easy answers are drawn from the least similar countries. Hard answers are drawn from the most similar ones. The number under each question is the average similarity of those three countries. `show` only prints the question; it does not choose the answers.

In [ ]:
def show(question):
    print(question.prompt)
    for option in question.options:
        print("-", option)
    print("answer:", question.answer)
    print("similarity of the wrong answers:", round(question.realized_difficulty, 3))
    print()

show(generate_question(countries, scores, "FRA", "capital", "easy", rng=np.random.default_rng(0)))
show(generate_question(countries, scores, "FRA", "capital", "hard", rng=np.random.default_rng(0)))

## Charts

The first chart counts the countries kept in each region. The second counts empty World Bank rows before the latest year is chosen. The third draws one easy, one medium, and one hard set of wrong answers for every country. The hard box should sit above the easy box.

In [ ]:
directory = figures_dir(root)
observations = load_indicators()
plot_countries_by_region(countries, directory / "countries_by_region.png")
plot_missing_indicators(observations, directory / "missing_indicators.png")
plot_difficulty_bands(scores, directory / "difficulty_bands.png")
for name in ("countries_by_region.png", "missing_indicators.png", "difficulty_bands.png"):
    display(Image(filename=str(directory / name)))

## Score

A correct answer is worth `1 + difficulty`. A wrong answer is worth 0. The session score is `100 * earned / maximum`. Missing the hard question costs more than missing the easy one. In the example the difficulties are 0.2, 0.8, and 0.5.

In [ ]:
missed_hard = session_score([(True, 0.2), (False, 0.8), (True, 0.5)])
missed_easy = session_score([(False, 0.2), (True, 0.8), (True, 0.5)])
print("Miss the hard question:", missed_hard)
print("Miss the easy question instead:", round(missed_easy, 1))